In [ ]:
from pathlib import Path
import glob
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns


In [ ]:
colors = [ "#828282","#C8C8C8", "#616161"]
custom_cmap = sns.color_palette(colors)

## Combine rprop and CellCycle classification

In [ ]:
in_path = '/Users/josefin/PhD/weihua_UHRF1_CellCycle/'

model = 'resnet18'

segmentation_subdirectory = 'results/segmentation_nuclei_IllCorr'

images_subdirectory = 'images/2603011/tif_IllCorr'

cellcycle_subdirectory = 'cellcycle_classification_retrained'

rprop_subdirectory = 'results/region_properties_nuclei_IllCorr'

In [ ]:
mask_files = sorted((Path(in_path) / segmentation_subdirectory).glob('[!.]*.tif'))

image_files = sorted((Path((in_path + images_subdirectory)).glob('*ch0.tif')))

cellcycle_files = sorted((Path((in_path + cellcycle_subdirectory)).glob('*.csv')))

rprop_files = sorted((Path((in_path + rprop_subdirectory)).glob('*.csv')))

len(mask_files) == len(image_files) == len(cellcycle_files) == len(rprop_files)


In [ ]:
out_subdirectory = 'combined_IllCorr_retrained_' + model

outdir = Path(in_path) / out_subdirectory

if not outdir.exists():
    outdir.mkdir()

In [ ]:
def calculate_sPhase_progression (df):
    df['s-phase_prog'] = np.nan
    df['s-phase_prog_corrected'] = np.nan
    df['s-phase-prob-sum'] = np.nan
    sphases = ['EarlyS', 'MidS', 'LateS']
    for idx, i in enumerate(df['max_class_name']):
        df.loc[idx, 's-phase-prob-sum'] = df['prob_EarlyS'][idx] + df['prob_MidS'][idx] + df['prob_LateS'][idx] + df['prob_G1G2'][idx] + df['prob_ambiguous'][idx]
        if i in sphases:
            df.loc[idx, 's-phase_prog'] = 0*df['prob_EarlyS'][idx] + 0.5*df['prob_MidS'][idx] + 1*df['prob_LateS'][idx]
            df.loc[idx, 's-phase_prog_corrected'] = 0 * (df['prob_EarlyS'][idx]/df['s-phase-prob-sum'][idx]) + 0.5 * (df['prob_MidS'][idx]/df['s-phase-prob-sum'][idx]) + 1*(df['prob_LateS'][idx]/df['s-phase-prob-sum'][idx])
        else:
            df.loc[idx, 's-phase_prog'] = np.nan


    return df


In [ ]:
for cc_file, rprop_file in zip(cellcycle_files, rprop_files):

    cc_df = pd.read_csv(cc_file)

    prop_df = pd.read_csv(rprop_file)
    prop_df['cell_id'] = prop_df['label']

    combined = cc_df.merge(prop_df,
                        on=['cell_id', 'mask_file'],
                        )
                        
    combined = calculate_sPhase_progression(combined)

    combined['genotype'] = str()
    for idx, i in enumerate(combined['img_file']):
        if i.startswith('wt'):
            combined.loc[idx, 'genotype'] = 'wt'
        elif i.startswith('1G12'):
            combined.loc[idx, 'genotype'] = '1G12'
        elif i.startswith('1E3'):
            combined.loc[idx, 'genotype'] = '1_E3' # underscore to avoid confusion with "1 to the power of 3"

    # combined.to_csv(outdir / (rprop_file.stem + '_combined.csv'))

## Plot & Measure

In [ ]:
outpath = '/Users/josefin/PhD/weihua_UHRF1_CellCycle/plots/'

In [ ]:
all_combined_files = sorted(Path('/Users/josefin/PhD/weihua_UHRF1_CellCycle/results/combined_IllCorr_retrained_resnet18').glob('*.csv'))

for file in all_combined_files:
    df = pd.read_csv(file)
    df_resnet = pd.concat([df_resnet, df], ignore_index=True) if 'df_resnet' in locals() else df

In [ ]:
## Distribution of sphase progression vs mean intensity for each genotype

fig, ax = plt.subplots(3,1,figsize=[8,8], sharex=True, sharey=True)
sns.scatterplot(df_resnet[df_resnet['genotype'] == 'wt'], 
           x='s-phase_prog_corrected', y='intensity_mean', 
           palette=custom_cmap, hue='max_class_name', hue_order=['EarlyS', 'MidS', 'LateS'],
           legend=True, ax=ax[0], alpha=0.6)
sns.scatterplot(df_resnet[df_resnet['genotype'] == '1_E3'], 
           x='s-phase_prog_corrected', y='intensity_mean', 
           palette=custom_cmap, hue='max_class_name', hue_order=['EarlyS', 'MidS', 'LateS'],
           legend=False, ax=ax[1], alpha=0.6)
sns.scatterplot(df_resnet[df_resnet['genotype'] == '1G12'], 
           x='s-phase_prog_corrected', y='intensity_mean', 
           palette=custom_cmap, hue='max_class_name', hue_order=['EarlyS', 'MidS', 'LateS'],
           legend=False, ax=ax[2], alpha=0.6)
ax[0].legend(title='Predicted S-Phase')
ax[0].set_ylabel('mean UHRF1 Intensity / Cell')
ax[1].set_ylabel('mean UHRF1 Intensity / Cell')
ax[2].set_ylabel('mean UHRF1 Intensity / Cell')
ax[0].set_title('WT')
ax[1].set_title('1E3')
ax[2].set_title('1G12')
ax[2].set_xlabel('S-Phase Progression')
ax[0].axvline(0.3, linestyle='--', c="#434343")
ax[1].axvline(0.3, linestyle='--', c='#434343')
ax[2].axvline(0.3, linestyle='--', c='#434343')
plt.tight_layout()

# plt.savefig((outpath + 'meanInt_sphaseProg.png'), dpi=300, transparent=False)

In [ ]:
df_resnet['phase'] = str()
for idx, i in enumerate(df_resnet['s-phase_prog_corrected']):
    if i < 0.3:
        df_resnet.loc[idx, 'phase'] = 'early'
    elif i >= 0.3:
        df_resnet.loc[idx, 'phase'] = 'late' # includes mid and late S-phase
    else:
        if np.isnan(i):
            max_class = df_resnet['max_class_name'][idx]
            if max_class == 'G1G2':
                df_resnet.loc[idx, 'phase'] = 'G1G2'
            else:
                df_resnet.loc[idx, 'phase'] = 'ambiguous'


In [ ]:
## previously, I had filtered the dataframe to only include early and mid/late S-phases

# df_sphases = df_resnet[(df_resnet['phase'] == 'early') | (df_resnet['phase'] == 'late')]
# df_sphases.head()

In [ ]:
df_sphases_G1G2 = df_resnet[(df_resnet['phase'] == 'G1G2') | (df_resnet['phase'] == 'early') | (df_resnet['phase'] == 'late')]
df_sphases_G1G2.head()

In [ ]:
df_sphases_G1G2 = df_sphases_G1G2.copy()

# Single wild-type mean (across all s-phases)
wt_global_median = (
    df_sphases_G1G2.loc[df_sphases_G1G2['genotype'] == 'wt', 'intensity_mean']
    .median()
)

# Normalize every row by that single value
df_sphases_G1G2.loc[:, 'intensity_norm_wt'] = (
    df_sphases_G1G2['intensity_mean'] / wt_global_median
)


In [ ]:
# df_sphases = df_resnet[(df_resnet['sphase'] == 'early') | (df_resnet['sphase'] == 'late')]

fig, ax = plt.subplots(figsize=(4,4))
plt.grid(True, which='both')
sns.boxplot(data=df_sphases_G1G2, x='genotype', y='intensity_norm_wt', hue='phase', hue_order=['G1G2', 'early', 'late'], 
            palette=custom_cmap, fliersize=1.5, )
plt.xlabel('Genotype')
plt.ylabel('normalized UHRF1 Intensity [AU]')
plt.yticks(np.arange(0, 11, 1))
plt.legend(title='Phase', loc='best', facecolor=None, edgecolor='black', fontsize=9, title_fontsize=10)

# plt.savefig((outpath + 'mean_UHRF1_norm_boxplot1.pdf'), dpi=300, transparent=True)



In [ ]:

fig, ax = plt.subplots(figsize=(6, 5))
sns.boxplot(data=df_sphases_G1G2, x='genotype', y='intensity_norm_wt', hue='phase', hue_order=['G1G2', 'early', 'late'], 
            palette='dark:black', fliersize=2.5, saturation=0.5, fill=False, linewidth=1.5, legend=False) 
plt.xlabel('Genotype')
plt.ylabel('normalized UHRF1 Intensity [AU]')
plt.legend(title='S-Phase', loc='best');
sns.swarmplot(
    data=df_sphases_G1G2,
    x='genotype',
    y='intensity_norm_wt',
    hue='phase',
    hue_order=['G1G2', 'early', 'late'],
    palette=custom_cmap,
    size=1,
    ax=ax,
    dodge=True,
    edgecolor='black',

)

ax.grid(True)
ax.set_xlabel('Genotype')
ax.set_ylabel('normalized UHRF1 Intensity [AU]')
ax.legend(title='S-Phase', loc='best')

plt.tight_layout()
plt.grid(True)

# plt.savefig(outpath + 'mean_UHRF1_norm_swarmplot.pdf', dpi=300, transparent=True)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(4, 4))

plt.hist(df_sphases_G1G2[df_sphases_G1G2['genotype'] == 'wt']['intensity_norm_wt'], bins=25, color="#D8AD11", alpha=0.1, histtype='bar')
plt.hist(df_sphases_G1G2[df_sphases_G1G2['genotype'] == 'wt']['intensity_norm_wt'], bins=25, color="#D8AD11", alpha=1, label='WT', histtype='step')

plt.hist(df_sphases_G1G2[df_sphases_G1G2['genotype'] == '1G12']['intensity_norm_wt'], bins=25, color="#CE56CA", alpha=0.2, histtype='bar')
plt.hist(df_sphases_G1G2[df_sphases_G1G2['genotype'] == '1G12']['intensity_norm_wt'], bins=25, color="#CE56CA", alpha=1, label='1G12', histtype='step')

plt.hist(df_sphases_G1G2[df_sphases_G1G2['genotype'] == '1_E3']['intensity_norm_wt'], bins=25, color="#22BEBE", alpha=0.2, histtype='bar')
plt.hist(df_sphases_G1G2[df_sphases_G1G2['genotype'] == '1_E3']['intensity_norm_wt'], bins=25, color="#22BEBE", alpha=1, label='1E3', histtype='step')

plt.xlabel('normalized UHRF1 Intensity [AU]')
plt.ylabel('Frequency')

plt.legend()

plt.tight_layout()
# plt.show()


# plt.savefig('/Users/josefin/PhD/weihua_UHRF1_CellCycle/plots/mean_UHRF1_norm_all_cmy.pdf', dpi=300, transparent=True)

## Statistics

In [ ]:
phase_counts = df_sphases_G1G2.groupby('phase').size()
genotype_counts = df_sphases_G1G2.groupby('genotype').size()

print("Phase counts:")
print(phase_counts)

print("\nGenotype counts:")
print(genotype_counts)

In [ ]:
# # Grouped summary statistics
grouped = df_sphases_G1G2.groupby(['genotype', 'phase'])['intensity_norm_wt']
median = grouped.median().unstack()
std = grouped.std().unstack()
print(f'Median: \n {median} \n Standard Deviation:\n {std}')

In [ ]:
from scipy.stats import mannwhitneyu
import itertools

results_sphase = []

for genotype in df_sphases_G1G2['genotype'].unique():
    subset = df_sphases_G1G2[df_sphases_G1G2['genotype'] == genotype]
    sphases = subset['phase'].unique()
    
    for s1, s2 in itertools.combinations(sphases, 2):
        g1 = subset[subset['phase'] == s1]['intensity_mean']
        g2 = subset[subset['phase'] == s2]['intensity_mean']
        
        stat, p = mannwhitneyu(g1, g2, alternative='two-sided')
        
        results_sphase.append((genotype, s1, s2, p))


In [ ]:
results_vs_wt = []

for sphase in df_sphases_G1G2['phase'].unique():
    wt = df_sphases_G1G2[
        (df_sphases_G1G2['genotype'] == 'wt') #&
        # (df_sphases_G1G2['phase'] == sphase)
    ]['intensity_mean']

    for genotype in df_sphases_G1G2['genotype'].unique():
        if genotype == 'wt':
            continue
        
        g = df_sphases_G1G2[
            (df_sphases_G1G2['genotype'] == genotype) #&
            # (df_sphases_G1G2['phase'] == sphase)
        ]['intensity_mean']
        
        stat, p = mannwhitneyu(wt, g, alternative='two-sided')
        
        results_vs_wt.append((sphase, genotype, p)) 


In [ ]:
from statsmodels.stats.multitest import multipletests

pvals = [r[3] for r in results_sphase]
_, pvals_corr_sphase, _, _ = multipletests(pvals, method='fdr_bh')
pvals = [r[2] for r in results_vs_wt]
_, pvals_corr_vs_wt, _, _ = multipletests(pvals, method='fdr_bh')


In [ ]:
def significance_label(p):
    if p < 0.0001:
        return '****'
    elif p < 0.001:
        return '***'
    elif p < 0.01:
        return '**'
    elif p < 0.05:
        return '*'
    return 'ns'


In [ ]:
print("\nCORRECTED Between S-phases")
for r, p  in zip(results_sphase, pvals_corr_sphase):
    print(f"{r[0]}: {r[1]} vs {r[2]} -> p={p:.3e}")
    print(significance_label(p))


print("\nCORRECTED Genotype vs WT")
for r, p  in zip(results_vs_wt, pvals_corr_vs_wt):
    print(f"{r[0]}: WT vs {r[1]} -> p={p:.3e}")
    print(significance_label(p))
